# Medical Assistant — Graph RAG + Agentic AI
### Powered by Google Gemini (FREE!)

**What this project does:**
- Builds a **Knowledge Graph** of diseases, symptoms, medicines
- Uses **Graph RAG** + **Vector Embeddings** for smart search
- Uses **Agentic Gemini AI** to reason and give structured reports
- Supports **PDF upload** to expand the knowledge base
- Includes **Model Evaluation** to measure accuracy

**Tech Stack:** Python · NetworkX · Sentence Transformers · Google Gemini · Gradio

---
###  Run each cell from top to bottom!

##  Step 1 — Install Libraries

In [ ]:
!pip install google-genai networkx matplotlib gradio PyPDF2 sentence-transformers --quiet
print(' All libraries installed!')

## Step 2 — Set Your FREE Google Gemini API Key

1. Go to **https://aistudio.google.com/**
2. Sign in with your Google account
3. Click **'Get API Key'** → **'Create API Key'** → Copy it

 No credit card needed!

In [ ]:
from google import genai

#  Get your FREE key at: https://aistudio.google.com/
# Click 'Get API Key' → 'Create API Key' → paste below
API_KEY = "paste-your-gemini-key-here"

client = genai.Client(api_key=API_KEY)

# Test connection
response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Say: API connected successfully!"
)
print(response.text)

## Step 3 — Build the Medical Knowledge Graph

This is the **Graph RAG** part!

- 🔴 **Red nodes** = Diseases
- 🟢 **Green nodes** = Medicines
- 🔵 **Blue nodes** = Symptoms / Side Effects
- **Arrows** = Relationships (has_symptom, treated_by, causes, side_effect)

In [ ]:
import networkx as nx

G = nx.DiGraph()

medical_edges = [
    # DIABETES
    ("Diabetes", "Frequent Urination", "has_symptom"),
    ("Diabetes", "Excessive Thirst", "has_symptom"),
    ("Diabetes", "Blurred Vision", "has_symptom"),
    ("Diabetes", "Fatigue", "has_symptom"),
    ("Diabetes", "Slow Healing Wounds", "has_symptom"),
    ("Diabetes", "Metformin", "treated_by"),
    ("Diabetes", "Insulin", "treated_by"),
    ("Metformin", "Nausea", "side_effect"),
    ("Metformin", "Diarrhea", "side_effect"),
    ("Metformin", "Alcohol", "interacts_with"),
    ("Insulin", "Low Blood Sugar", "side_effect"),
    ("Diabetes", "Kidney Disease", "causes"),

    # HYPERTENSION
    ("Hypertension", "Headache", "has_symptom"),
    ("Hypertension", "Dizziness", "has_symptom"),
    ("Hypertension", "Chest Pain", "has_symptom"),
    ("Hypertension", "Blurred Vision", "has_symptom"),
    ("Hypertension", "Amlodipine", "treated_by"),
    ("Hypertension", "Lisinopril", "treated_by"),
    ("Amlodipine", "Swollen Ankles", "side_effect"),
    ("Lisinopril", "Dry Cough", "side_effect"),
    ("Hypertension", "Heart Attack", "causes"),

    # INFLUENZA
    ("Influenza", "Fever", "has_symptom"),
    ("Influenza", "Body Ache", "has_symptom"),
    ("Influenza", "Fatigue", "has_symptom"),
    ("Influenza", "Cough", "has_symptom"),
    ("Influenza", "Sore Throat", "has_symptom"),
    ("Influenza", "Paracetamol", "treated_by"),
    ("Paracetamol", "Liver Damage", "side_effect"),
    ("Paracetamol", "Ibuprofen", "interacts_with"),

    # MIGRAINE
    ("Migraine", "Severe Headache", "has_symptom"),
    ("Migraine", "Nausea", "has_symptom"),
    ("Migraine", "Light Sensitivity", "has_symptom"),
    ("Migraine", "Blurred Vision", "has_symptom"),
    ("Migraine", "Sumatriptan", "treated_by"),
    ("Migraine", "Ibuprofen", "treated_by"),
    ("Sumatriptan", "Dizziness", "side_effect"),
    ("Ibuprofen", "Stomach Ulcer", "side_effect"),

    # ASTHMA
    ("Asthma", "Wheezing", "has_symptom"),
    ("Asthma", "Shortness of Breath", "has_symptom"),
    ("Asthma", "Chest Tightness", "has_symptom"),
    ("Asthma", "Cough", "has_symptom"),
    ("Asthma", "Salbutamol", "treated_by"),
    ("Salbutamol", "Tremors", "side_effect"),
    ("Salbutamol", "Rapid Heartbeat", "side_effect"),

    # DEPRESSION
    ("Depression", "Persistent Sadness", "has_symptom"),
    ("Depression", "Fatigue", "has_symptom"),
    ("Depression", "Sleep Problems", "has_symptom"),
    ("Depression", "Loss of Appetite", "has_symptom"),
    ("Depression", "Sertraline", "treated_by"),
    ("Depression", "Fluoxetine", "treated_by"),
    ("Sertraline", "Nausea", "side_effect"),
    ("Sertraline", "Insomnia", "side_effect"),
    ("Sertraline", "Alcohol", "interacts_with"),
    ("Fluoxetine", "Anxiety", "side_effect"),
]

for src, tgt, rel in medical_edges:
    G.add_edge(src, tgt, relation=rel)

print('Knowledge Graph built!')
print(f'   Nodes: {G.number_of_nodes()}')
print(f'   Edges: {G.number_of_edges()}')

## Step 4 — Visualise the Knowledge Graph

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

diseases  = ["Diabetes", "Hypertension", "Influenza", "Migraine", "Asthma", "Depression"]
medicines = ["Metformin", "Insulin", "Amlodipine", "Lisinopril",
             "Paracetamol", "Sumatriptan", "Ibuprofen", "Salbutamol",
             "Sertraline", "Fluoxetine"]

color_map = []
for node in G.nodes():
    if node in diseases:      color_map.append('#e74c3c')
    elif node in medicines:   color_map.append('#2ecc71')
    else:                     color_map.append('#3498db')

plt.figure(figsize=(20, 14))
pos = nx.spring_layout(G, seed=42, k=2.2)
nx.draw(G, pos,
        node_color=color_map, node_size=1400,
        with_labels=True, font_size=7, font_weight='bold',
        arrows=True, edge_color='#bdc3c7', alpha=0.9)

plt.legend(handles=[
    Patch(color='#e74c3c', label='🔴 Disease'),
    Patch(color='#2ecc71', label='🟢 Medicine'),
    Patch(color='#3498db', label='🔵 Symptom / Side Effect'),
], loc='upper left', fontsize=12)

plt.title('Medical Knowledge Graph', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()
print('Graph visualised!')

## 🔍 Step 5 — Graph RAG Search Functions

**Graph RAG** follows connections in the graph:

Symptom → Disease → Medicine → Side Effects

In [ ]:
def graph_search_by_symptoms(symptom_list):
    results = {}
    all_diseases = ["Diabetes", "Hypertension", "Influenza", "Migraine", "Asthma", "Depression"]

    for disease in all_diseases:
        disease_symptoms = [
            tgt for _, tgt, data in G.out_edges(disease, data=True)
            if data['relation'] == 'has_symptom'
        ]
        matched = [
            ds for us in symptom_list for ds in disease_symptoms
            if us.lower() in ds.lower() or ds.lower() in us.lower()
        ]
        if not matched:
            continue

        medicines_found = [
            tgt for _, tgt, data in G.out_edges(disease, data=True)
            if data['relation'] == 'treated_by'
        ]
        medicine_details = {}
        for med in medicines_found:
            medicine_details[med] = {
                'side_effects': [
                    tgt for _, tgt, data in G.out_edges(med, data=True)
                    if data['relation'] == 'side_effect'
                ],
                'interactions': [
                    tgt for _, tgt, data in G.out_edges(med, data=True)
                    if data['relation'] == 'interacts_with'
                ]
            }
        complications = [
            tgt for _, tgt, data in G.out_edges(disease, data=True)
            if data['relation'] == 'causes'
        ]
        results[disease] = {
            'match_score': len(matched),
            'matched_symptoms': matched,
            'medicines': medicine_details,
            'complications': complications
        }
    return dict(sorted(results.items(), key=lambda x: x[1]['match_score'], reverse=True))


def graph_search_medicine(medicine_name):
    result = {}
    for node in G.nodes():
        if medicine_name.lower() in node.lower():
            result[node] = {
                'used_for': [
                    src for src, tgt, d in G.in_edges(node, data=True)
                    if d['relation'] == 'treated_by'
                ],
                'side_effects': [
                    tgt for _, tgt, d in G.out_edges(node, data=True)
                    if d['relation'] == 'side_effect'
                ],
                'interactions': [
                    tgt for _, tgt, d in G.out_edges(node, data=True)
                    if d['relation'] == 'interacts_with'
                ]
            }
    return result


print(' Graph RAG search ready!')
test = graph_search_by_symptoms(['Headache', 'Dizziness'])
for disease, info in list(test.items())[:2]:
    print(f'  → {disease}: matched {info["match_score"]} symptom(s)')

## Step 6 — Real Vector Embeddings (Actual RAG!)

**Normal RAG** = keyword search (ctrl+F)

**Real RAG** = understands MEANING using vectors

Example: searches for *'waking up to pee'* finds *'Frequent Urination'* → *'Diabetes'*

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print(' Loading embedding model...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model loaded!')

# Add patient lay terms to graph
lay_terms = [
    ("Frequent Urination", "waking up to pee", "also_known_as"),
    ("Frequent Urination", "peeing a lot", "also_known_as"),
    ("Excessive Thirst", "always thirsty", "also_known_as"),
    ("Headache", "head is pounding", "also_known_as"),
    ("Dizziness", "feel dizzy", "also_known_as"),
    ("Persistent Sadness", "feeling low", "also_known_as"),
    ("Loss of Appetite", "lost interest in everything", "also_known_as"),
    ("Chest Tightness", "chest feels tight", "also_known_as"),
    ("Wheezing", "whistling sound breathing", "also_known_as"),
    ("Severe Headache", "bad headache one side", "also_known_as"),
    ("Light Sensitivity", "can't stand light", "also_known_as"),
    ("Nausea", "feeling sick", "also_known_as"),
    ("Fatigue", "always tired", "also_known_as"),
    ("Blurred Vision", "can't see clearly", "also_known_as"),
]
for src, tgt, rel in lay_terms:
    G.add_edge(src, tgt, relation=rel)

# Create embeddings for all nodes
node_list = list(G.nodes())
node_texts = [str(node) for node in node_list]
node_embeddings = embed_model.encode(node_texts)

print(f'Created embeddings for {len(node_list)} nodes!')
print(f'   Each node = vector of {node_embeddings.shape[1]} numbers')
print(f'   Graph now has {G.number_of_nodes()} nodes')

##  Step 7 — Vector RAG Search Function

In [ ]:
def real_rag_search(query, top_k=5):
    """
    REAL RAG search using vector embeddings!
    Finds nodes by MEANING not just keywords.
    """
    query_embedding = embed_model.encode([query])[0]
    similarities = []
    for i, node in enumerate(node_list):
        similarity = np.dot(query_embedding, node_embeddings[i]) / (
            np.linalg.norm(query_embedding) * np.linalg.norm(node_embeddings[i])
        )
        similarities.append((similarity, node))

    similarities.sort(reverse=True)
    top_nodes = [node for _, node in similarities[:top_k]]

    results = {}
    for node in top_nodes:
        connections = {'similar_to_query': node, 'connected_to': []}
        for _, neighbor, data in G.out_edges(node, data=True):
            connections['connected_to'].append({'node': neighbor, 'relation': data['relation']})
        for src, _, data in G.in_edges(node, data=True):
            connections['connected_to'].append({'node': src, 'relation': f"is_{data['relation']}_of"})
        results[node] = connections
    return results


print(' Vector RAG search ready!')
print('\n Test — searching for "lung pain breathing difficulty":')
test = real_rag_search('lung pain breathing difficulty')
for node, info in list(test.items())[:3]:
    print(f'  → Found: "{node}"')
    print(f'    Connected to: {[c["node"] for c in info["connected_to"][:3]]}')

##  Step 8 — Agentic Gemini AI

Gemini AI **autonomously decides**:
- When to use keyword search vs vector search
- How many times to search
- How to combine results and give final answer

In [ ]:
import json
from google.genai import types

def medical_agent_v2(user_query):
    """Agentic loop with Graph RAG + Vector RAG!"""
    print(f'\n Patient: {user_query}')
    print(' Agent thinking...\n' + '─'*60)

    SYSTEM_PROMPT = """You are an expert medical assistant AI powered by a medical knowledge graph with real vector embeddings.

When a user describes symptoms:
1. ALWAYS use search_by_symptoms tool first
2. ALWAYS use real_vector_search tool to find semantically similar conditions
3. Combine both results to give a comprehensive answer
4. Rank diseases by likelihood
5. If they ask about a medicine use search_medicine_info

Always structure your answer like this:

🔍 SYMPTOM ANALYSIS
 POSSIBLE CONDITIONS (ranked by likelihood)
 TREATMENT OPTIONS
 IMPORTANT WARNINGS
 RECOMMENDATION

Always end with: Please consult a real doctor for diagnosis."""

    tools = [
        types.Tool(function_declarations=[
            types.FunctionDeclaration(
                name="search_by_symptoms",
                description="Search medical knowledge graph using symptoms — keyword matching.",
                parameters=types.Schema(
                    type="OBJECT",
                    properties={"symptoms": types.Schema(type="ARRAY", items=types.Schema(type="STRING"), description="List of symptoms")},
                    required=["symptoms"]
                )
            ),
            types.FunctionDeclaration(
                name="real_vector_search",
                description="Search using vector embeddings — finds by MEANING not keywords.",
                parameters=types.Schema(
                    type="OBJECT",
                    properties={"query": types.Schema(type="STRING", description="Natural language symptom description")},
                    required=["query"]
                )
            ),
            types.FunctionDeclaration(
                name="search_medicine_info",
                description="Get detailed info about a medicine — side effects, interactions.",
                parameters=types.Schema(
                    type="OBJECT",
                    properties={"medicine_name": types.Schema(type="STRING", description="Medicine name")},
                    required=["medicine_name"]
                )
            )
        ])
    ]

    messages = [types.Content(role="user", parts=[types.Part(text=user_query)])]

    for step in range(8):
        response = client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=messages,
            config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, tools=tools)
        )

        has_tool_call = False
        for part in response.candidates[0].content.parts:
            if part.function_call is not None and part.function_call.name:
                has_tool_call = True
                tool_name = part.function_call.name
                tool_args = dict(part.function_call.args)
                print(f'🔧 Using tool: {tool_name} | Input: {tool_args}')

                if tool_name == 'search_by_symptoms':
                    result = graph_search_by_symptoms(tool_args.get('symptoms', []))
                elif tool_name == 'real_vector_search':
                    result = real_rag_search(tool_args.get('query', ''))
                elif tool_name == 'search_medicine_info':
                    result = graph_search_medicine(tool_args.get('medicine_name', ''))
                else:
                    result = {}

                print(f'   Done!')
                messages.append(response.candidates[0].content)
                messages.append(types.Content(
                    role="tool",
                    parts=[types.Part(function_response=types.FunctionResponse(
                        name=tool_name,
                        response={"result": json.dumps(result, indent=2)}
                    ))]
                ))
                break

        if not has_tool_call:
            final = response.text
            print(final)
            return final

    return 'Could not complete analysis.'

print(' Medical Agent v2 ready!')

##  Step 9 — PDF Knowledge Base

Upload any medical PDF to expand the knowledge graph with real data!

Recommended: Download free PDFs from **who.int**

In [ ]:
import PyPDF2
import re

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"
    return text

def extract_entities_from_chunk(chunk, chunk_num):
    prompt = f"""You are a medical knowledge graph builder.
Extract medical entities and relationships from the text below.

Rules:
- Return ONLY a JSON array of arrays, like: [[\"Disease\",\"Symptom\",\"has_symptom\"], ...]
- Each inner array must have exactly 3 strings: [source, target, relationship]
- Only use these relationships: has_symptom, treated_by, causes, side_effect, interacts_with
- Extract at least 10 relationships if possible
- No explanation, no markdown, no code fences — ONLY the JSON array

Text chunk {chunk_num}:
{chunk}
"""
    response = client.models.generate_content(model="gemini-2.5-flash-lite", contents=prompt)
    return response.text.strip()

def add_pdf_to_graph(pdf_path):
    print(f" Reading PDF: {pdf_path}")
    text = extract_text_from_pdf(pdf_path)
    print(f" Extracted {len(text)} characters")
    chunk_size = 6000
    chunks = [text[i:i+chunk_size] for i in range(0, min(len(text), chunk_size * 5), chunk_size)]
    print(f" Processing {len(chunks)} chunks...")
    total_added = 0
    for i, chunk in enumerate(chunks):
        print(f"  Chunk {i+1}/{len(chunks)}...", end=" ")
        try:
            raw = extract_entities_from_chunk(chunk, i+1)
            clean = re.sub(r"```[a-z]*", "", raw).strip().rstrip("`").strip()
            edges = json.loads(clean)
            added = 0
            for item in edges:
                if isinstance(item, list) and len(item) == 3:
                    src, tgt, rel = item
                    G.add_edge(str(src).strip(), str(tgt).strip(), relation=str(rel).strip())
                    added += 1
            total_added += added
            print(f" +{added} edges")
        except Exception as e:
            print(f" Skipped — {e}")
    # Rebuild embeddings after adding new nodes
    global node_list, node_texts, node_embeddings
    node_list = list(G.nodes())
    node_texts = [str(node) for node in node_list]
    node_embeddings = embed_model.encode(node_texts)
    print(f"\n Done! Added {total_added} edges.")
    print(f" Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

print("PDF functions ready!")

##  Step 10 — Upload Your PDF (Optional)

Skip this step if you don't have a PDF.

Tip: Download free medical PDFs from **https://www.who.int/publications**

In [ ]:
from google.colab import files

print(" Upload your medical PDF (or skip this cell)")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n Processing: {filename}")
    add_pdf_to_graph(filename)
    print(f"\n Graph now has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

## Step 11 — Model Performance Evaluation

This measures how accurate our model is on:
- **Easy test** — standard medical terms (100% expected)
- **Hard test** — vague real-world patient language (harder!)

In [ ]:
# ── EASY TEST ──────────────────────────────────────────────────────
print('EASY TEST — Standard medical terminology')
print('─' * 60)

easy_tests = [
    {"symptoms": "frequent urination, excessive thirst, blurred vision", "expected": "Diabetes"},
    {"symptoms": "severe headache, dizziness, chest pain", "expected": "Hypertension"},
    {"symptoms": "fever, body ache, sore throat, runny nose", "expected": "Influenza"},
    {"symptoms": "severe headache, nausea, light sensitivity", "expected": "Migraine"},
    {"symptoms": "wheezing, shortness of breath, chest tightness", "expected": "Asthma"},
    {"symptoms": "persistent sadness, fatigue, sleep problems", "expected": "Depression"},
    {"symptoms": "fatigue, slow healing wounds, excessive thirst", "expected": "Diabetes"},
    {"symptoms": "headache, blurred vision, shortness of breath", "expected": "Hypertension"},
    {"symptoms": "cough, fever, fatigue, body ache", "expected": "Influenza"},
    {"symptoms": "wheezing, cough, chest tightness", "expected": "Asthma"},
]

correct_easy = 0
for i, test in enumerate(easy_tests):
    symptoms_list = test["symptoms"].split(", ")
    graph_results = graph_search_by_symptoms(symptoms_list)
    predicted = list(graph_results.keys())[0] if graph_results else "Unknown"
    is_correct = test["expected"].lower() in predicted.lower()
    if is_correct: correct_easy += 1
    print(f"{'✅' if is_correct else '❌'} Test {i+1}: Expected={test['expected']}, Got={predicted}")

print(f"\n Easy Test Accuracy: {correct_easy}/10 = {correct_easy*10}%")

# ── HARD TEST ──────────────────────────────────────────────────────
print('\n' + '─' * 60)
print(' HARD TEST — Vague real-world patient language')
print('─' * 60)

hard_tests = [
    {"symptoms": "I keep waking up at night to pee and I am always hungry", "expected": "Diabetes"},
    {"symptoms": "my head is pounding and I feel dizzy when I stand up", "expected": "Hypertension"},
    {"symptoms": "I feel really low, can't sleep and lost interest in everything", "expected": "Depression"},
    {"symptoms": "my chest feels tight in the morning and I make a whistling sound", "expected": "Asthma"},
    {"symptoms": "bad headache on one side with vomiting and can't stand light", "expected": "Migraine"},
]

correct_hard = 0
for i, test in enumerate(hard_tests):
    symptoms_list = test["symptoms"].split()
    graph_results = graph_search_by_symptoms(symptoms_list)
    vector_results = real_rag_search(test["symptoms"], top_k=3)
    predicted_graph = list(graph_results.keys())[0] if graph_results else "Unknown"
    predicted_vector = list(vector_results.keys())[0] if vector_results else "Unknown"
    is_correct = (
        test["expected"].lower() in predicted_graph.lower() or
        test["expected"].lower() in predicted_vector.lower()
    )
    if is_correct: correct_hard += 1
    status = '✅ if is_correct else '❌'
    print(f"{status} Test {i+1}: Expected={test['expected']}")
    print(f"   Graph RAG: {predicted_graph} | Vector RAG: {predicted_vector}")

print(f"\n Hard Test Accuracy: {correct_hard}/5 = {correct_hard*20}%")
print('\n' + '─' * 60)
print(f"\n FINAL SUMMARY:")
print(f"   Easy test (medical terms):     {correct_easy*10}%")
print(f"   Hard test (patient language):  {correct_hard*20}%")
if correct_hard*20 < 60:
    print("\n To improve: Upload more medical PDFs to expand the knowledge graph!")

##  Step 12 — Launch the App!

In [ ]:
import gradio as gr
import matplotlib.patches as mpatches

def draw_graph():
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    fig.patch.set_facecolor('#161c2e')
    ax.set_facecolor('#161c2e')
    type_map = {}
    for u, v, d in G.edges(data=True):
        rel = d.get('relation', '')
        if rel == 'treated_by':  type_map[u] = 'disease';  type_map[v] = 'medicine'
        elif rel == 'has_symptom': type_map[u] = 'disease'; type_map[v] = 'symptom'
        elif rel == 'side_effect': type_map[v] = 'side_effect'
    for n in G.nodes():
        if n not in type_map: type_map[n] = 'symptom'
    color_map2 = {'disease': '#ff4d6d', 'medicine': '#00ffb2', 'symptom': '#7b61ff', 'side_effect': '#ffbe0b'}
    node_colors = [color_map2.get(type_map.get(n, 'symptom'), '#7b61ff') for n in G.nodes()]
    node_sizes  = [800 if type_map.get(n) == 'disease' else 500 if type_map.get(n) == 'medicine' else 300 for n in G.nodes()]
    pos = nx.spring_layout(G, k=2.5, iterations=50, seed=42)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#ffffff15', arrows=True, arrowsize=8, width=0.8)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=node_sizes, alpha=0.95)
    nx.draw_networkx_labels(G, pos, labels={n: n for n in G.nodes() if type_map.get(n) == 'disease'}, ax=ax, font_size=9, font_color='#ffffff', font_weight='bold')
    ax.legend(handles=[
        mpatches.Patch(color='#ff4d6d', label='Disease'),
        mpatches.Patch(color='#00ffb2', label='Medicine'),
        mpatches.Patch(color='#7b61ff', label='Symptom'),
        mpatches.Patch(color='#ffbe0b', label='Side Effect'),
    ], loc='upper left', facecolor='#0f1320', edgecolor='#ffffff1a', labelcolor='#dde3f0', fontsize=8)
    ax.set_title(f'Medical Knowledge Graph  ·  {G.number_of_nodes()} nodes  ·  {G.number_of_edges()} edges', color='#5a6280', fontsize=9, pad=10)
    ax.axis('off')
    plt.tight_layout()
    return fig

def run_agent_ui(query):
    if not query.strip(): return "❌ Please enter your symptoms!"
    try: return medical_agent_v2(query)
    except Exception as e: return f"Error: {str(e)}"

custom_css = """
body, .gradio-container { background: #080b12 !important; }
footer { display: none !important; }
textarea, input[type='text'] { background: #161c2e !important; border: 1px solid #ffffff1a !important; border-radius: 12px !important; color: #dde3f0 !important; }
.gr-button-primary { background: linear-gradient(135deg, #00ffb2, #00cc8f) !important; border: none !important; border-radius: 12px !important; color: #040810 !important; font-weight: 600 !important; }
"""

with gr.Blocks(css=custom_css, title="MedGraph AI") as demo:
    gr.HTML(f"""
    <div style="background:linear-gradient(90deg,#080b12,#0d1220);border-bottom:1px solid #ffffff0f;padding:16px 28px;display:flex;align-items:center;justify-content:space-between;margin-bottom:16px">
      <div style="display:flex;align-items:center;gap:12px">
        <div style="width:34px;height:34px;border-radius:9px;background:linear-gradient(135deg,#00ffb2,#7b61ff);display:flex;align-items:center;justify-content:center;font-size:16px">⚕</div>
        <div>
          <div style="font-size:15px;font-weight:600;color:#dde3f0">MedGraph AI</div>
          <div style="font-size:10px;color:#5a6280;font-family:monospace">Graph RAG · Vector Embeddings · Agentic AI · Gemini</div>
        </div>
      </div>
      <div style="display:flex;gap:10px">
        <span style="font-family:monospace;font-size:10px;padding:4px 10px;border-radius:20px;border:1px solid #00ffb260;color:#00ffb2">{G.number_of_nodes()} nodes · {G.number_of_edges()} edges</span>
        <span style="font-family:monospace;font-size:10px;padding:4px 10px;border-radius:20px;border:1px solid #7b61ff60;color:#7b61ff">Real Embeddings </span>
      </div>
    </div>
    """)
    with gr.Row():
        with gr.Column(scale=1):
            graph_plot = gr.Plot(label="KNOWLEDGE GRAPH", show_label=True)
            demo.load(fn=draw_graph, outputs=graph_plot)
        with gr.Column(scale=1):
            query_input = gr.Textbox(lines=5, placeholder="Describe your symptoms...\ne.g. persistent cough with blood, chest pain, weight loss", label="DESCRIBE YOUR SYMPTOMS")
            run_btn = gr.Button("Run Medical Analysis ↗", variant="primary")
            gr.Examples(examples=[
                ["Persistent cough with blood, chest pain and weight loss."],
                ["Severe headache, nausea and sensitivity to light."],
                ["Always tired, very thirsty, blurry vision."],
                ["Wheezing, chest tightness, shortness of breath."],
                ["Persistent sadness, sleep problems, loss of interest."],
                ["Side effects of Metformin?"],
            ], inputs=query_input, label="QUICK EXAMPLES")
            output = gr.Textbox(lines=18, label="AI MEDICAL REPORT", interactive=False)
    run_btn.click(fn=run_agent_ui, inputs=query_input, outputs=output)

demo.launch(share=True)
print(' App launched!')